# MemoryBank: Long-Term Memory for LLMs with Ebbinghaus Forgetting Curve
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/weilinear/paper_reading/blob/main/notebooks/03_memory_bank_demo.ipynb)

Interactive demonstration of **MemoryBank** (Zhong et al., AAAI 2024 / arXiv:2305.10250):
- 3-Tiered Storage (Dialogue log, Hierarchical Event Summary, User Personality Portrait).
- Ebbinghaus Forgetting Curve ($R = e^{-t / S}$) & Spacing Effect Reinforcement ($S \leftarrow S + 1, t \leftarrow 0$).
- Dynamic Prompt Assembly for the SiliconFriend AI Companion.


In [ ]:
# 1. Setup Environment
![ -d paper_reading ] || git clone https://github.com/weilinear/paper_reading.git
%cd paper_reading


In [ ]:
# 2. Visualize Ebbinghaus Forgetting Curves with Matplotlib
import numpy as np
import matplotlib.pyplot as plt

days = np.linspace(0, 10, 100)
plt.figure(figsize=(9, 5))

# Plot forgetting curves for different memory strengths
for S in [1.0, 2.0, 3.0, 4.0]:
    R = np.exp(-days / S)
    label = f"S = {S:.0f} ({'Initial memory' if S == 1 else f'Recalled {int(S-1)} time(s)'})"
    plt.plot(days, R, label=label, linewidth=2)

plt.axhline(y=0.15, color='red', linestyle='--', label='Forgetting Threshold (R = 0.15)')
plt.title("Ebbinghaus Forgetting Curve & Spacing Effect in MemoryBank", fontsize=13)
plt.xlabel("Elapsed Time t (Days since last recall)", fontsize=11)
plt.ylabel("Memory Retention R = exp(-t / S)", fontsize=11)
plt.ylim(0, 1.05)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=10)
plt.show()


In [ ]:
# 3. Run the Multi-Day MemoryBank Interactive Simulation
from memory_bank import MemoryBank

# Initialize MemoryBank for Linda
bank = MemoryBank(forgetting_threshold=0.15, user_name="Linda")

# Day 1: Add initial conversations
bank.record_turn(1.0, "I want to learn Python. What book do you recommend?", "I suggest 'Automate the Boring Stuff with Python'.")
bank.record_turn(1.0, "I had a ham sandwich for lunch today.", "Sounds tasty! Enjoy your break.")
bank.event_summary.add_daily_summary(1, "Linda asked for Python learning resources (book: Automate the Boring Stuff).")

# Day 2: Emotional interaction & User portrait update
bank.record_turn(2.0, "I feel stressed at work and anxious about learning.", "Take it step by step; you're doing great.")
bank.user_portrait.traits = ["introverted", "ambitious", "growth-oriented"]
bank.user_portrait.interests = ["Python programming", "reading"]
bank.user_portrait.emotional_state = "experiencing workplace learning anxiety"
bank.event_summary.add_daily_summary(2, "Linda expressed workplace learning stress and received encouragement.")

print("Memory Status on Day 2.0:")
for m in bank.memories:
    print(f"• [{m.id}] S={m.strength:.1f}, R={m.get_retention(2.0):.2f}: {m.content[:45]}...")


In [ ]:
# 4. Day 4: Query triggers the Spacing Effect!
# Query recalling the book on Day 4.0
query = "What was the Python book you recommended?"
retrieved = bank.retrieve(query, current_day=4.0, top_k=1, apply_forgetting=False)

recalled_mem = retrieved[0][0]
print(f"✓ Recalled: [{recalled_mem.id}] -> New Strength S = {recalled_mem.strength:.1f} (Reinforced!)")


In [ ]:
# 5. Day 7: Divergence between Reinforced vs Unreinforced Memories
print("Memory Retention Comparison on Day 7.0:")
for m in bank.memories:
    r = m.get_retention(7.0)
    status = "RETAINED" if r >= bank.forgetting_threshold else "FORGOTTEN"
    print(f"• [{m.id}] S={m.strength:.1f}, R={r:.4f} -> {status} ({m.content[:40]}...)")


In [ ]:
# 6. Final Prompt Assembled for the LLM Companion (SiliconFriend)
prompt = bank.format_prompt("Can you remind me of that Python book? How should I study it?", current_day=7.0)
print(prompt)
